In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

class AdaptiveGatingCrossAttention(layers.Layer):
    """
    Implements the Adaptive Gating Module (AGM) and Cross-Attention
    described in Section 3.3.1.2 (Equations 8-11).
    """
    def __init__(self, d_model, dropout_rate=0.1):
        super(AdaptiveGatingCrossAttention, self).__init__()
        self.d_model = d_model

        # Projections for Gating Mechanism (Eq 8)
        self.W_Qg = layers.Dense(d_model)
        self.W_Kg = layers.Dense(d_model)

        # Projections for Masks (Eq 9, 10)
        self.W_Qm = layers.Dense(d_model, activation='sigmoid')
        self.W_Km = layers.Dense(d_model, activation='sigmoid')

        # Standard Attention Projections
        self.W_Q = layers.Dense(d_model)
        self.W_K = layers.Dense(d_model)
        self.W_V = layers.Dense(d_model)

        self.dropout = layers.Dropout(dropout_rate)
        self.softmax = layers.Softmax(axis=-1)

    def call(self, inputs):
        # inputs = [query_source, key_value_source]
        # For X_img (Image Attends to Text): query=Image, key=Text
        Q_src, K_src = inputs

        # 1. Prepare Query, Key, Value (Eq 4, 5)
        Q = self.W_Q(Q_src)
        K_proj = self.W_K(K_src)
        V = self.W_V(K_src)

        # 2. Adaptive Gating Module (AGM)
        # Fusion (Eq 8) - Note: Dimensions must match, broadcasting might be needed if seq len differs
        # Assuming Q and K are projected to same d_model space
        # For distinct sequence lengths, we perform fusion via broadcasting or tiling.
        # However, standard AGM usually operates on the interaction space.
        # Given the paper's description: G = (Q.Wg + bg) * (K.Wg + bg) (Element-wise)
        # This implies shape alignment. If seq lengths differ (100 vs 100), it works directly.

        G_Q = self.W_Qg(Q)
        G_K = self.W_Kg(K_proj)
        G = G_Q * G_K # Element-wise product (Eq 8)

        # Gating Masks (Eq 9, 10)
        M_Q = self.W_Qm(G)
        M_K = self.W_Km(G)

        # Apply Gates
        Q_gated = M_Q * Q
        K_gated = M_K * K_proj

        # 3. Scaled Dot Product Attention (Eq 11)
        # Score = Q . K^T / sqrt(d)
        matmul_qk = tf.matmul(Q_gated, K_gated, transpose_b=True)
        dk = tf.cast(tf.shape(K_gated)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        attention_weights = self.softmax(scaled_attention_logits)
        output = tf.matmul(attention_weights, V)

        return output

class ContextAwareEmbedding(layers.Layer):
    """
    Fuses Region Features with Position Embeddings.
    Section 3.3.1.3 (Equations 12, 13).
    """
    def __init__(self, target_dim):
        super(ContextAwareEmbedding, self).__init__()
        self.pos_proj = layers.Dense(target_dim, activation='sigmoid') # Eq 12

    def call(self, region_feats, pos_feats):
        # region_feats: (B, 100, 768) (already projected to target_dim)
        # pos_feats: (B, 100, 6) (normalized coords/area as per paper)

        # Project position features (Eq 12)
        I_pos = self.pos_proj(pos_feats)

        # Element-wise multiplication (Eq 13)
        F_img_context = region_feats * I_pos
        return F_img_context

class MANM(models.Model):
    def __init__(self, dropout_rate=0.3):
        super(MANM, self).__init__()

        # Dimensions defined in paper
        self.txt_dim = 768  # BERT
        self.img_dim = 1024 # Mask R-CNN
        self.projection_dim = 768 # Align image to text (Eq 3)
        self.final_seq_dim = 256 # Dimension after MSAN (Eq 16 implicitly/Output)

        # --- 1. Align Dimensions ---
        self.img_projector = layers.Dense(self.projection_dim) # Eq 3

        # --- 2. Context Aware Embedding ---
        self.context_embedding = ContextAwareEmbedding(self.projection_dim)

        # --- 3. MCAN (Cross Attention) ---
        # Image attending to Text
        self.cross_att_img = AdaptiveGatingCrossAttention(self.projection_dim)
        # Text attending to Image
        self.cross_att_txt = AdaptiveGatingCrossAttention(self.projection_dim)

        # --- 4. MSAN (Self Attention) ---
        # Paper uses Multi-head Self Attention
        self.msan_img = layers.MultiHeadAttention(num_heads=4, key_dim=self.projection_dim)
        self.msan_txt = layers.MultiHeadAttention(num_heads=4, key_dim=self.projection_dim)

        # Final projection to 256 dim as per Eq 16 description/Diagram
        self.final_proj_img = layers.Dense(self.final_seq_dim, activation='relu')
        self.final_proj_txt = layers.Dense(self.final_seq_dim, activation='relu')

        # --- 5. Pooling & Concatenation ---
        self.global_max_pool = layers.GlobalMaxPooling1D()
        self.concat = layers.Concatenate()
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, inputs):
        # inputs: [text_embeddings, image_embeddings, image_pos_embeddings]
        # text_embeddings: (B, 100, 768)
        # image_embeddings: (B, 100, 1024)
        # image_pos_embeddings: (B, 100, 6)

        F_txt, F_img_raw, I_pos_raw = inputs

        # 1. Project Image Features (Eq 3)
        # F_proj_img: (B, 100, 768)
        F_img_proj = self.img_projector(F_img_raw)

        # 2. Context-Aware Fusion (Eq 13)
        # Apply Context awareness using position embeddings
        # F_img_context: (B, 100, 768)
        F_img_context = self.context_embedding(F_img_proj, I_pos_raw)

        # 3. Multimodal Context-aware Attention (MCAN)
        # Cross-Attended Image Features (X_img): Query=Image, Key=Text, Value=Text
        X_img = self.cross_att_img([F_img_context, F_txt])

        # Cross-Attended Text Features (X_txt): Query=Text, Key=Image, Value=Image
        X_txt = self.cross_att_txt([F_txt, F_img_context])

        # 4. Multihead Self-Attention (MSAN) (Eq 14, 16)
        # Self-attend image features
        X_img_sa = self.msan_img(X_img, X_img)
        # Self-attend text features
        X_txt_sa = self.msan_txt(X_txt, X_txt)

        # Project to final dimension (256)
        X_img_final = self.final_proj_img(X_img_sa)
        X_txt_final = self.final_proj_txt(X_txt_sa)

        # 5. Pooling and Concatenation (Eq 17)
        # Max Pooling over sequence dimension (100 -> 1)
        # Output: (B, 256)
        pool_img = self.global_max_pool(X_img_final)
        pool_txt = self.global_max_pool(X_txt_final)

        # Concatenate: (B, 512)
        joint_representation = self.concat([pool_img, pool_txt])

        return joint_representation

# --- Example Usage / Mock Data ---
if __name__ == "__main__":
    # Parameters based on paper
    BATCH_SIZE = 32
    SEQ_LEN = 100
    TXT_DIM = 768
    IMG_DIM = 1024
    POS_DIM = 6 # (x, y, w, h, area, aspect_ratio)

    # Mock Inputs
    text_emb = tf.random.normal((BATCH_SIZE, SEQ_LEN, TXT_DIM))
    img_emb = tf.random.normal((BATCH_SIZE, SEQ_LEN, IMG_DIM))
    pos_emb = tf.random.normal((BATCH_SIZE, SEQ_LEN, POS_DIM))

    # Initialize Model
    manm_module = MANM()

    # Forward Pass
    output = manm_module([text_emb, img_emb, pos_emb])

    print("Input Text Shape:", text_emb.shape)
    print("Input Image Shape:", img_emb.shape)
    print("MANM Output Shape:", output.shape) # Should be (32, 512)